# Practica Etiquetador — Notebook de estudio

> **Autor:** Alejandro Marinas
> **Asignatura:** Aprenentatge Automatic — La Salle Gracia
> **Repositorio:** [github.com/MarinasAlejandro/practica-etiquetador](https://github.com/MarinasAlejandro/practica-etiquetador)

Este notebook recorre paso a paso el proyecto: la **teoria** que hay detras de cada decision y el **codigo** que la implementa. Cada concepto se explica primero con palabras y se demuestra despues con codigo ejecutable, usando los datos reales del proyecto.

**Como usarlo:** lee la celda de markdown, ejecuta la celda de codigo siguiente con `Shift + Enter`, observa el resultado, y pasa a la siguiente.

## Indice

1. Que hace el sistema y por que dos modelos
2. Conceptos basicos: clasificar, supervisado, distancia, espacio de caracteristicas
3. Cargar el dataset y mirarlo
4. KNN para clasificar la forma
   1. Preprocesado: pasar a gris y aplanar
   2. Distancia euclidiana
   3. Votacion de los k vecinos
   4. Entrenar y evaluar el KNN
5. K-Means para detectar los colores
   1. Cada pixel como punto en RGB
   2. Inicializacion, asignacion, recalculo, convergencia
   3. Aplicar K-Means a una imagen real
6. find_bestK con el criterio del 20%
7. get_colors: de RGB a nombre de color
8. predict_image (end-to-end)
9. Retrievals: el buscador combinado
10. Los 4 analisis de eficiencia
11. Matriz de confusion: donde acierta y donde falla
12. Resumen final y posibles preguntas del profe

## 1. Que hace el sistema y por que dos modelos

El proyecto construye un **etiquetador automatico** para una tienda online de ropa. Dada una imagen, el sistema le asigna dos tipos de etiquetas:

- La **forma de la prenda** (vestido, camisa, sandalia...) usando KNN supervisado.
- Los **colores predominantes** (rojo, azul, rosa...) usando K-Means no supervisado.

Sobre estas etiquetas se construye un **buscador combinado** que permite consultas tipo *"pink dress"*, y un **frontend web con Flask** que ofrece dos funcionalidades: subir una imagen para etiquetarla, y filtrar el catalogo por forma y color.

### Por que dos modelos distintos

| Aspecto | Forma | Color |
|---|---|---|
| Tipo | Supervisado | No supervisado |
| Algoritmo | KNN | K-Means |
| Hay etiquetas? | Si (gt.json) | No (cada imagen tiene una mezcla de colores sin pre-etiquetar) |
| Que hace | Aprende de ejemplos | Descubre patrones |

Un solo algoritmo no podria hacer ambas cosas: KNN no descubre patrones nuevos, K-Means no aprende de etiquetas.

### Cifras del dataset

- **2.328** imagenes de entrenamiento
- **851** imagenes de test
- **9** clases de forma (el PDF decia 8 pero el dataset incluye Heels)
- **11** colores basicos universales

## 2. Conceptos basicos

Antes de tocar codigo, hay que tener claro el vocabulario.

- **Clasificar**: asignar una categoria a un elemento nuevo. En este proyecto, decir si una imagen es Dresses, Shirts, Heels...
- **Clase / Etiqueta**: las categorias posibles. Aqui las 9 formas de prenda.
- **Aprendizaje supervisado**: el modelo aprende a partir de ejemplos ya etiquetados. Tu sabes la respuesta correcta y se la das.
- **Aprendizaje no supervisado**: no hay etiquetas. El modelo tiene que descubrir patrones por si solo.
- **Conjunto de entrenamiento (train)**: ejemplos con etiqueta que el modelo ve.
- **Conjunto de test**: imagenes nuevas que no ha visto durante el entrenamiento.
- **Caracteristica / Feature**: un numero que describe parte del elemento. Aqui, cada pixel en gris.
- **Espacio de caracteristicas**: el "mundo" matematico donde viven los datos. Si una imagen tiene 4.800 pixeles, vive en un espacio de 4.800 dimensiones.
- **Distancia euclidiana**: la generalizacion de la distancia "en linea recta" a muchas dimensiones.

### Pildora matematica: distancia euclidiana

Para dos puntos $A = (a_1, a_2, ..., a_n)$ y $B = (b_1, b_2, ..., b_n)$:

$$d(A, B) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

**Distancia pequena** = puntos parecidos. **Distancia grande** = puntos diferentes.

Vamos a verlo con un ejemplo simple:

In [ ]:
import numpy as np

# Dos puntos en 2 dimensiones
A = np.array([1, 2])
B = np.array([4, 6])

# Calculo manual
distancia = np.sqrt((A[0] - B[0])**2 + (A[1] - B[1])**2)
print(f"Distancia entre A={A} y B={B}: {distancia:.4f}")

# Forma vectorizada (la que usaremos en el proyecto)
distancia_vec = np.sqrt(np.sum((A - B)**2))
print(f"Distancia vectorizada: {distancia_vec:.4f}")

La forma vectorizada (`np.sqrt(np.sum((A - B)**2))`) es la que se usa en el proyecto porque funciona igual para 2 dimensiones que para 4.800. Es la base de todo: tanto el KNN como K-Means la usan.

## 3. Cargar el dataset y mirarlo

El profesor proporciona un archivo `utils_data.py` con la funcion `read_dataset()` que carga las imagenes ya redimensionadas a 80x60 y las etiquetas del `gt.json`.

In [ ]:
import numpy as np
from utils_data import read_dataset

# Cargar todo el dataset
train_imgs, train_class, train_color, test_imgs, test_class, test_color = read_dataset(
    root_folder='./images/',
    gt_json='./images/gt.json'
)

print(f"Train: {train_imgs.shape}")
print(f"  - Etiquetas de forma: {train_class.shape}")
print(f"  - Etiquetas de color (lista por imagen): {train_color.shape}")
print(f"Test:  {test_imgs.shape}")
print()
print(f"Clases de forma: {sorted(set(train_class))}")
print()
print(f"Ejemplo: imagen 0 es '{train_class[0]}' con colores {train_color[0]}")

Cada imagen es un array (80, 60, 3): 80 filas, 60 columnas, 3 canales (RGB). Las etiquetas de forma son strings (un valor por imagen) y las de color son listas (una imagen puede tener varios colores).

Vamos a visualizar algunas:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_imgs[i])
    ax.set_title(f"{train_class[i]}\n{train_color[i]}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. KNN para clasificar la forma

KNN significa **K Nearest Neighbours** = K vecinos mas cercanos. La idea es muy intuitiva:

> Guardas todas las imagenes de entrenamiento con sus etiquetas. Cuando llega una imagen nueva, miras las **k mas parecidas** (las mas cercanas en el espacio de caracteristicas) y votas la clase mayoritaria.

### Por que es supervisado

Necesitas el train con etiquetas reales. El modelo no "aprende reglas", solo guarda y compara.

### El parametro k

- **k pequeno** (k=1): te quedas con el vecino mas cercano. Sensible a outliers.
- **k grande** (k=11): mas robusto pero pierde detalle local.

En este proyecto se usa **k=3** porque equilibra accuracy (90.25%) y robustez. Con k=1 obtienes 91.89% pero un outlier puede contaminar la prediccion.

### 4.1 Preprocesado: pasar a gris y aplanar

Las imagenes vienen en 80x60x3 (RGB). El KNN no puede trabajar con eso directamente. Hay que convertirlas a un **vector de numeros**.

#### Paso 1: pasar a escala de grises

Para clasificar la **forma**, el color es ruido. Una camiseta roja y una azul tienen la misma silueta. Pasamos de 3 canales a 1 con la funcion `utils.rgb2gray` (proporcionada por el profe), que combina los 3 canales en uno solo aplicando los pesos clasicos de luminosidad:

$$gris = 0.2989 \cdot R + 0.5870 \cdot G + 0.1140 \cdot B$$

#### Paso 2: aplanar

Convertimos la matriz 80x60 en un vector de 4.800 numeros (un pixel = una dimension).

In [ ]:
import utils

# Coger una imagen de ejemplo
img = train_imgs[0]
print(f"Imagen original: shape={img.shape}, dtype={img.dtype}")
print(f"  - 80 filas x 60 columnas x 3 canales = {80*60*3} valores")

# Paso 1: pasar a gris
img_gris = utils.rgb2gray(img.astype(float))
print(f"\nDespues de rgb2gray: shape={img_gris.shape}")
print(f"  - 80 filas x 60 columnas (un solo canal) = {80*60} valores")

# Paso 2: aplanar
img_vector = img_gris.reshape(-1)
print(f"\nDespues de aplanar: shape={img_vector.shape}")
print(f"  - Vector de {img_vector.shape[0]} dimensiones")

Visualicemos el efecto del preprocesado:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title(f"Original RGB\n{img.shape}")
axes[0].axis('off')

axes[1].imshow(img_gris, cmap='gray')
axes[1].set_title(f"Gris\n{img_gris.shape}")
axes[1].axis('off')

axes[2].plot(img_vector[:200])
axes[2].set_title("Primeros 200 pixeles\ndel vector aplanado")
axes[2].set_xlabel("Indice")
axes[2].set_ylabel("Intensidad")
plt.tight_layout()
plt.show()

### 4.2 Distancia euclidiana entre imagenes

Una vez aplanadas, dos imagenes son dos puntos en un espacio de 4.800 dimensiones. La **distancia euclidiana** entre ellas mide cuan parecidas son.

In [ ]:
# Tres imagenes del train
img_a = train_imgs[0]
img_b = train_imgs[1]
img_c = train_imgs[2]

# Preprocesar las tres
def preprocesar(img):
    return utils.rgb2gray(img.astype(float)).reshape(-1)

a = preprocesar(img_a)
b = preprocesar(img_b)
c = preprocesar(img_c)

# Distancias
d_ab = np.sqrt(np.sum((a - b)**2))
d_ac = np.sqrt(np.sum((a - c)**2))
d_bc = np.sqrt(np.sum((b - c)**2))

print(f"Distancia A-B: {d_ab:>10.2f}  ({train_class[0]} vs {train_class[1]})")
print(f"Distancia A-C: {d_ac:>10.2f}  ({train_class[0]} vs {train_class[2]})")
print(f"Distancia B-C: {d_bc:>10.2f}  ({train_class[1]} vs {train_class[2]})")
print()
print("Observa: imagenes de la misma clase tienen distancias menores.")

### 4.3 Votacion de los k vecinos

Para clasificar una imagen nueva:

1. Calculamos su distancia a las 2.328 del train.
2. Ordenamos las distancias de menor a mayor.
3. Cogemos las **k=3** mas pequenas.
4. Miramos las etiquetas de esos 3 vecinos y votamos la clase mayoritaria.

In [ ]:
# Coger una imagen del test (que el modelo no ha visto)
test_idx = 0
img_test = test_imgs[test_idx]
forma_real = test_class[test_idx]

# Preprocesar el test y todo el train
test_vector = preprocesar(img_test)
train_vectors = np.array([preprocesar(im) for im in train_imgs])

# Distancia de la imagen test a TODAS las del train (vectorizado)
diferencias = train_vectors - test_vector
distancias = np.sqrt(np.sum(diferencias**2, axis=1))

# Coger los 3 mas cercanos
k = 3
indices_cercanos = np.argsort(distancias)[:k]
clases_cercanas = train_class[indices_cercanos]

print(f"Imagen test #{test_idx}: clase real = '{forma_real}'")
print(f"\nLos {k} vecinos mas cercanos:")
for i, idx in enumerate(indices_cercanos):
    print(f"  {i+1}. Train #{idx}: clase '{train_class[idx]}', distancia {distancias[idx]:.2f}")

# Votar la clase mayoritaria
clases, votos = np.unique(clases_cercanas, return_counts=True)
prediccion = clases[np.argmax(votos)]
print(f"\nPrediccion (clase mayoritaria): '{prediccion}'")
print(f"Acierta? {'SI' if prediccion == forma_real else 'NO'}")

### 4.4 Entrenar el KNN completo y medir accuracy

Toda la logica anterior esta encapsulada en la clase `KNN` del archivo `KNN.py`. Vamos a usarla con todo el test:

In [ ]:
from KNN import KNN
import time

# Entrenar
t0 = time.time()
knn = KNN(train_imgs, train_class)
t_entrenar = time.time() - t0
print(f"Entrenamiento: {t_entrenar:.2f}s")
print(f"  Shape de train preprocesado: {knn.train_data.shape}")

# Predecir todo el test con varios k
print("\nResultados:")
print(f"  {'k':>3}  {'accuracy':>10}  {'tiempo':>10}")
for k in [1, 3, 5]:
    t0 = time.time()
    pred = knn.predict(test_imgs, k=k)
    t = time.time() - t0
    acc = 100 * np.sum(pred == test_class) / len(test_class)
    print(f"  {k:>3}  {acc:>9.2f}%  {t:>9.2f}s")

**Resultado esperado:** k=1 da 91.89%, k=3 da 90.25%. El sistema usa k=3 por defecto en produccion porque es mas robusto a outliers.

## 5. K-Means para detectar los colores

K-Means es un algoritmo de **clustering** (agrupar datos similares) **no supervisado** (sin etiquetas).

### La idea clave para colores

Cada **pixel** es un punto en el espacio RGB de 3 dimensiones. Una imagen 80x60 tiene 4.800 pixeles, asi que es como tener 4.800 puntos en un espacio 3D.

K-Means agrupa esos puntos en **K clusters**. Cada centroide del cluster es una media de los pixeles que se le han asignado, y representa **un color predominante**.

### Los 4 pasos del algoritmo

1. **Inicializacion**: colocar K centroides (estrategias: 'first' o 'random').
2. **Asignacion**: cada pixel al centroide mas cercano.
3. **Recalculo**: cada centroide es la media de sus pixeles.
4. **Repetir** 2-3 hasta convergencia (los centroides no se mueven).

### 5.1 Visualizar los pixeles como puntos en RGB

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Coger una imagen y visualizar sus pixeles en 3D
img = train_imgs[100]
pixels = img.reshape(-1, 3)  # (4800, 3)

# Submuestreamos para que se vea claro (mostramos 500 pixeles)
np.random.seed(42)
sample = pixels[np.random.choice(len(pixels), 500, replace=False)]

fig = plt.figure(figsize=(12, 5))

# Imagen original
ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(img)
ax1.set_title(f"Imagen ({train_class[100]})")
ax1.axis('off')

# Pixeles en RGB
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.scatter(sample[:, 0], sample[:, 1], sample[:, 2], c=sample/255.0, s=20)
ax2.set_xlabel('R')
ax2.set_ylabel('G')
ax2.set_zlabel('B')
ax2.set_title("Pixeles como puntos en RGB\n(submuestra de 500)")
plt.tight_layout()
plt.show()

### 5.2 K-Means paso a paso

Vamos a ejecutar el algoritmo iteracion por iteracion para ver como se mueven los centroides:

In [ ]:
import Kmeans

# Aplicar KMeans con K=3 y ver iteraciones
img = train_imgs[100]
km = Kmeans.KMeans(img, K=3)
km.fit()

print(f"Convergio en {km.num_iter} iteraciones")
print(f"Centroides finales (RGB):")
for i, c in enumerate(km.centroids.astype(int)):
    print(f"  Cluster {i}: R={c[0]}, G={c[1]}, B={c[2]}")
print(f"\nWCD final: {km.withinClassDistance():.2f}")

### Visualizar el resultado: imagen original vs comprimida con K colores

In [ ]:
def comprimir_con_kmeans(img, K):
    """Aplica KMeans y reconstruye la imagen con solo K colores."""
    km = Kmeans.KMeans(img, K=K)
    km.fit()
    # Reemplazamos cada pixel por su centroide
    pixels_comprimidos = km.centroids[km.labels].reshape(img.shape).astype('uint8')
    return pixels_comprimidos, km.centroids

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(img)
axes[0].set_title("Original\n(miles de colores)")
axes[0].axis('off')

for i, K in enumerate([2, 3, 5, 8], start=1):
    img_comp, centroides = comprimir_con_kmeans(img, K)
    axes[i].imshow(img_comp)
    axes[i].set_title(f"K={K}\n{K} colores")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

Fijate como con K=2 la imagen pierde mucha informacion (solo 2 colores), y con K=8 se parece mucho a la original. **El truco esta en encontrar el K minimo que represente bien los colores predominantes.**

## 6. find_bestK con el criterio del 20%

¿Como decidimos automaticamente cuantos colores usar para cada imagen? El PDF (pag. 8) define este criterio:

1. Probar K = 2, 3, 4, ... hasta un maximo.
2. Calcular la **WCD (Within-Class Distance)** para cada K. La WCD es la media de las distancias al cuadrado de cada pixel a su centroide. Mide cuan compactos son los clusters.
3. **Cuanto mas K, menor WCD**, porque hay mas centroides para repartir los pixeles.
4. Cuando anadir un cluster mas reduce la WCD **menos del 20%**, paramos. Anadir mas no compensa.

Formula:

$$\text{decremento} = 100 - 100 \cdot \frac{WCD_k}{WCD_{k-1}}$$

Si decremento < 20%, nos quedamos con K-1.

In [ ]:
# Hacer la tabla manualmente para una imagen
img = train_imgs[100]
print(f"Imagen: {train_class[100]}, colores reales: {train_color[100]}")
print()
print(f"  {'K':>3}  {'WCD':>10}  {'decremento %':>13}  {'<20%?'}")
print("-" * 50)

wcd_anterior = None
mejor_K = None
for k in range(2, 9):
    km = Kmeans.KMeans(img, K=k)
    km.fit()
    wcd = km.withinClassDistance()
    if wcd_anterior is None:
        print(f"  {k:>3}  {wcd:>10.2f}  {'-':>13}  -")
    else:
        decremento = 100 - 100 * (wcd / wcd_anterior)
        marca = "<- aqui paramos" if (decremento < 20 and mejor_K is None) else ""
        print(f"  {k:>3}  {wcd:>10.2f}  {decremento:>12.2f}%  {marca}")
        if decremento < 20 and mejor_K is None:
            mejor_K = k - 1
    wcd_anterior = wcd

print(f"\nK optima encontrada: {mejor_K}")

Lo mismo se hace en una funcion del proyecto: `KMeans.find_bestK(max_K)`. Vamos a verificar que da el mismo resultado:

In [ ]:
km = Kmeans.KMeans(img, K=2)
mejor_K_funcion = km.find_bestK(max_K=8)
print(f"find_bestK devuelve: {mejor_K_funcion}")
print(f"Centroides con K={mejor_K_funcion}:")
for c in km.centroids.astype(int):
    print(f"  RGB: ({c[0]}, {c[1]}, {c[2]})")

## 7. get_colors: de RGB a nombre de color

Los centroides son vectores RGB de 3 numeros. Pero el sistema tiene que devolver nombres legibles como "Pink" o "Blue". Para eso:

- El profe nos da una funcion `utils.get_color_prob(rgb)` que devuelve un vector de **11 probabilidades** (una por cada color basico universal).
- Nuestra funcion `get_colors` coge el `argmax` de esas probabilidades para cada centroide.

Los 11 colores son: Red, Orange, Brown, Yellow, Green, Blue, Purple, Pink, Black, Grey, White.

In [ ]:
# Probar get_colors con la imagen anterior
nombres = Kmeans.get_colors(km.centroids)
print(f"Centroides RGB y nombres asignados:")
for centroide, nombre in zip(km.centroids.astype(int), nombres):
    print(f"  RGB ({centroide[0]:>3}, {centroide[1]:>3}, {centroide[2]:>3}) -> {nombre}")

print(f"\nColores reales del ground truth: {train_color[100]}")

Observa que los colores predichos suelen incluir los del ground truth, mas algunos del fondo (blanco/gris). Es la limitacion que K-Means tiene al analizar toda la imagen sin distinguir el producto del fondo.

## 8. predict_image: el etiquetador end-to-end

La funcion `predict_image` junta KNN + K-Means + get_colors. Le pasas una imagen y te devuelve forma + colores predominantes:

In [ ]:
import my_labeling

# Probar con varias imagenes del test
for idx in [0, 100, 500]:
    img = test_imgs[idx]
    resultado = my_labeling.predict_image(img, knn)

    print(f"Imagen test #{idx}:")
    print(f"  Real:     forma={test_class[idx]}, colores={test_color[idx]}")
    print(f"  Predicho: forma={resultado['shape']}, colores={resultado['colors']}, K={resultado['K']}")
    print()

## 9. Retrievals: el buscador combinado

Las funciones de retrieval en `my_labeling.py` filtran un conjunto de imagenes por color, forma o ambos:

- `retrieval_by_color(images, color_labels, query_color)`
- `retrieval_by_shape(images, shape_labels, query_shape)`
- `retrieval_combined(images, color_labels, shape_labels, query_color, query_shape)` -> AND logico

In [ ]:
# Buscar todos los vestidos rosas en el test
imgs_pink_dress, indices = my_labeling.retrieval_combined(
    test_imgs,
    test_color,   # etiquetas de color
    test_class,   # etiquetas de forma
    query_color='pink',
    query_shape='dress'
)

print(f"Encontrados {len(indices)} vestidos rosas en el test")
print()

# Mostrar los primeros 8
n_mostrar = min(8, len(indices))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < n_mostrar:
        idx = indices[i]
        ax.imshow(test_imgs[idx])
        ax.set_title(f"{test_class[idx]}\n{test_color[idx]}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 10. Los 4 analisis de eficiencia

El enunciado pide minimo 3 analisis. Hicimos 4 para cubrir distintos aspectos del sistema. Los resultados completos estan en `informe/results.json`.

In [ ]:
import json

with open('informe/results.json') as f:
    resultados = json.load(f)

# Mostrar resumen de los 4 analisis
print("=" * 60)
print("ANALISIS 1: Inicializacion de centroides (K=4, N=50 imagenes)")
print("=" * 60)
for est, datos in resultados['analisis_1_inicializacion'].items():
    print(f"  {est:<8}  iter={datos['iter_media']:.2f}  WCD={datos['wcd_media']:.2f}  t={datos['t_medio_ms']:.1f}ms")

In [ ]:
print("=" * 60)
print("ANALISIS 2: Valor de K en KNN (851 imagenes test)")
print("=" * 60)
for k, datos in resultados['analisis_2_k_knn'].items():
    print(f"  k={k:>2}  accuracy={datos['accuracy']:.2f}%  t={datos['tiempo_s']:.2f}s")

In [ ]:
print("=" * 60)
print("ANALISIS 3: Criterio find_bestK (umbrales sobre 30 imagenes)")
print("=" * 60)
for umbral, datos in resultados['analisis_3_criterio_bestK'].items():
    print(f"  umbral {umbral:>3}%:  K media={datos['K_media']:.2f}  K min={datos['K_min']}  K max={datos['K_max']}")

In [ ]:
print("=" * 60)
print("ANALISIS 4: Normalizacion Min-Max en el KNN (k=3)")
print("=" * 60)
for config, datos in resultados['analisis_4_normalizacion'].items():
    print(f"  {config:<18}  accuracy={datos['accuracy']:.2f}%  t={datos['tiempo_s']:.2f}s")

### Conclusiones de cada analisis

1. **Inicializacion**: 'random' converge ~10% mas rapido que 'first', con WCD final practicamente igual. Diferencia marginal. Se usa 'first' por defecto (determinista).

2. **K en KNN**: k=1 es el mejor (91.89%) pero sensible a outliers. k=3 (90.25%) es el optimo en produccion por robustez. El accuracy decrece en general al aumentar k (con un ligero repunte en k=7).

3. **Criterio find_bestK**: a mayor umbral, menor K elegida. El 20% del PDF da K media = 5.60, equilibrio razonable. Con 10% sobreajusta (K=6.90), con 30% pierde detalle (K=4.40).

4. **Normalizacion Min-Max**: NO cambia el accuracy. Las features (pixeles) ya estan en el mismo rango [0, 255]. La normalizacion es clave cuando hay variables con escalas distintas, no aqui. Esto valida indirectamente la decision del PDF de pasar a gris.

## 11. Matriz de confusion: donde acierta y donde falla

El accuracy global (90.25% con k=3) esconde matices importantes. Hagamos una matriz de confusion para ver que clases se confunden:

In [ ]:
# Predecir todo el test con k=3
pred = knn.predict(test_imgs, k=3)

# Accuracy por clase
clases = sorted(set(test_class))
print(f"{'Clase':<14}  {'Total':>6}  {'Aciertos':>9}  {'Accuracy':>9}")
print("-" * 48)
for c in clases:
    mask = test_class == c
    total = mask.sum()
    aciertos = ((pred == c) & mask).sum()
    acc = 100 * aciertos / total
    print(f"{c:<14}  {total:>6}  {aciertos:>9}  {acc:>8.1f}%")

In [ ]:
# Confusiones graves (mas del 5% de las imagenes de una clase)
print("Confusiones graves (>5% de los casos):")
print()
for c_real in clases:
    mask = test_class == c_real
    total = mask.sum()
    confusiones = []
    for c_pred in clases:
        if c_pred == c_real:
            continue
        n = ((pred == c_pred) & mask).sum()
        if n > 0 and 100*n/total > 5:
            confusiones.append((c_pred, n, 100*n/total))
    if confusiones:
        print(f"  {c_real} (n={total}):")
        for c_pred, n, pct in sorted(confusiones, key=lambda x: -x[1]):
            print(f"    -> {c_pred}: {n} casos ({pct:.1f}%)")

### Observacion clave

Las **unicas confusiones graves** se dan entre los **3 tipos de calzado abierto**: Heels (tacones), Sandals (sandalias) y Flip Flops (chanclas). Visualmente son casi identicos en una foto 80x60 en gris.

**Lo que NO pasa:** el sistema NO confunde calcetines con pantalones, ni vestidos con camisas, ni bolsos con nada. Las clases muy distintas estan bien separadas en el espacio de pixeles.

Esto es un comportamiento esperable y honesto de un KNN basado en pixeles.

## 12. Resumen final y posibles preguntas del profe

### Datos clave que tienes que recordar

- **Accuracy global con k=3:** 90.25%
- **Mejor accuracy (k=1):** 91.89%
- **K media de find_bestK con criterio 20%:** 5.60
- **Clases que mas falla:** Heels (77.8%), Sandals (81.9%) -> calzado abierto
- **Clases que mejor funcionan:** Bolsos, Pantalones, Camisas (>95%)

### Preguntas tipicas y como contestar

| Pregunta | Respuesta corta |
|---|---|
| ¿Que hace tu sistema? | Etiquetador automatico de imagenes de ropa: KNN para forma, K-Means para color, buscador combinado |
| ¿Por que dos modelos? | Forma es supervisada, color no supervisado: problemas distintos |
| ¿Por que pasas a gris? | Para clasificar forma el color es ruido. Reduce 14.400 dims a 4.800 |
| ¿Que k usas? | k=3 en produccion. k=1 da mas accuracy pero es sensible a outliers |
| ¿Como eliges el K de KMeans? | find_bestK con criterio del 20% del PDF sobre WCD |
| ¿Como conviertes RGB a nombre? | utils.get_color_prob (proporcionada) + argmax |
| ¿Que clases confunde? | Solo entre los 3 calzados abiertos. No confunde categorias muy distintas |
| ¿Cuando NO usarias el sistema? | Decisiones automaticas con coste real (stock, devoluciones), dominios criticos |

### Limitaciones conocidas

1. Fondos no uniformes confunden a K-Means (coge colores del fondo).
2. No hay "no se": si una prenda no esta en las 9 clases, la asigna igualmente.
3. El KNN es sensible a la pose y la escala (compara pixel a pixel).
4. El criterio del 20% es heuristico, no optimo en todos los casos.

---

## Para profundizar

- **Codigo fuente completo:** [github.com/MarinasAlejandro/practica-etiquetador](https://github.com/MarinasAlejandro/practica-etiquetador)
- **Informe completo:** `informe/informe.md` (con desarrollo, analisis y conclusiones)
- **Frontend Flask:** ejecutar `python app/app.py` y abrir http://127.0.0.1:5001